# Corner Detection Module

This notebook focuses on detecting and analyzing **corner features** in images using two classical algorithms:

- **Harris Corner Detection**  
- **Shi–Tomasi Corner Detection**

The main objectives are:

1. To detect corner points from **preprocessed images**.  
2. To compare the behavior of the two algorithms across different images.  
3. To evaluate the effect of **algorithm parameters** on corner detection results.  

This module forms a crucial part of the feature extraction pipeline, providing insights into image structure and preparing for subsequent shape analysis tasks.

## Importing Required Libraries

In this cell, we import the essential Python libraries for our corner detection module:

- **`cv2` (OpenCV)**: Provides image processing functions such as reading images, converting color spaces, and performing corner detection.  
- **`numpy`**: Supports numerical operations on image arrays, including matrix computations and manipulation of pixel values.  
- **`matplotlib.pyplot`**: Used for visualizing images and plotting results.  
- **`os`**: Enables interaction with the file system, such as listing image files in a directory.

These libraries form the foundation for loading, processing, and visualizing images in our corner detection pipeline.

In [2]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

## Setting Up Image Directory

In this step, we define the location of the dataset images and ensure the directory is correctly loaded.

- **`IMAGE_DIR`** is the relative path to the `images/` folder in the project root.  
  Since this notebook is located in `individual work/member3/`, we move **two levels up** (`../../`) to reach the root and then access the `images/` folder.

- We verify the directory exists using `os.path.exists()`. If the folder is missing, a `FileNotFoundError` is raised to prevent errors later in the pipeline.

- We create a list of all image files in the folder with extensions `.png`, `.jpg`, or `.jpeg`.

- Finally, we print the **total number of images found** and display a few **sample file names** to confirm that the directory is correctly loaded and accessible.

In [4]:


IMAGE_DIR = "../../images"

# Verify directory exists
if not os.path.exists(IMAGE_DIR):
    raise FileNotFoundError(f"Directory '{IMAGE_DIR}' not found. Check relative path.")

image_files = [
    f for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith(('.png', '.jpg', '.jpeg'))
]

print(f"Total images found: {len(image_files)}")
print("Sample files:", image_files[:3])

Total images found: 15
Sample files: ['shape1.jpg', 'building1.jpg', 'shape5.jpg']


## 1. Preprocessing Integration

Corner detection algorithms are highly sensitive to **noise** and **intensity fluctuations** in images.  

To improve robustness and ensure stable corner detection, we apply the following preprocessing steps:

- **Grayscale Conversion**: Converts the image to a single intensity channel, simplifying computation and reducing unnecessary color information.  
- **Gaussian Smoothing**: Applies a Gaussian blur to reduce high-frequency noise while preserving important structural features.  

The resulting **blurred grayscale image** is then used as input for both Harris and Shi–Tomasi corner detection methods, ensuring consistent and reliable results across all images.

In [5]:
def preprocess_image(img_path):
    img = cv2.imread(img_path)

    if img is None:
        print(f"Warning: Could not read image {img_path}")
        return None, None

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    return img, blur

## 2. Harris Corner Detection

Harris corner detection is based on the **structure tensor matrix**, which captures local gradients in an image.

The **corner response function** is defined as:

$$
R = \det(M) - k \, (\mathrm{trace}(M))^2
$$

Where:

- $\det(M) = \lambda_1 \lambda_2$  
- $\mathrm{trace}(M) = \lambda_1 + \lambda_2$  
- $k \in [0.04, 0.06]$  

Here, $\lambda_1$ and $\lambda_2$ are the eigenvalues of the structure tensor at each pixel.  

A **large positive value of $R$** indicates a corner. This allows the algorithm to distinguish corners from edges and flat regions.  

In [6]:
def harris_corner_detection(image, gray, k=0.04, threshold_ratio=0.01):
    gray = np.float32(gray)

    dst = cv2.cornerHarris(gray, 2, 3, k)
    dst = cv2.dilate(dst, None)

    threshold = threshold_ratio * dst.max()
    corner_mask = dst > threshold

    corner_count = np.sum(corner_mask)

    image_harris = image.copy()
    image_harris[corner_mask] = [255, 0, 0]

    return image_harris, corner_count

## 3. Shi–Tomasi Corner Detection

The **Shi–Tomasi Corner Detection** algorithm improves upon Harris by selecting only the strongest corners.  

- For each pixel, compute the eigenvalues $\lambda_1$ and $\lambda_2$ of the structure tensor matrix.  
- The corner response is defined as:

$$
R = \min(\lambda_1, \lambda_2)
$$

- A pixel is considered a **corner** if $R$ exceeds a specified threshold.  

Shi–Tomasi tends to produce **fewer, more stable corners** compared to Harris, making it particularly effective for applications such as feature tracking and shape analysis.  

In this notebook, we use OpenCV’s `goodFeaturesToTrack()` function to implement Shi–Tomasi corner detection efficiently.

In [7]:
def shi_tomasi_detection(image, gray, max_corners=100, quality=0.01, min_dist=10):
    image_copy = image.copy()

    corners = cv2.goodFeaturesToTrack(
        gray,
        maxCorners=max_corners,
        qualityLevel=quality,
        minDistance=min_dist
    )

    corner_count = 0

    if corners is not None:
        corner_count = len(corners)
        for corner in corners:
            x, y = corner.ravel()
            cv2.circle(image_copy, (int(x), int(y)), 4, (0, 255, 0), -1)

    return image_copy, corner_count

## 4. Applying Both Methods

In this step, we process all images stored in the `images/` folder located in the project root.  

For each image, we compute:

- **Harris corner count** – the number of corners detected by the Harris algorithm.  
- **Shi–Tomasi corner count** – the number of corners detected by the Shi–Tomasi algorithm.  
- **Difference (Harris - Shi–Tomasi)** – a simple quantitative comparison between the two methods.  

This analysis allows us to objectively compare both algorithms across different images, evaluate their behavior, and understand how each method responds to image complexity and features.

In [8]:
for file in image_files:
    path = os.path.join(IMAGE_DIR, file)

    image, blur = preprocess_image(path)

    if image is None:
        continue

    harris_img, harris_count = harris_corner_detection(image.copy(), blur)
    shi_img, shi_count = shi_tomasi_detection(image.copy(), blur)

    print(f"\nImage: {file}")
    print(f"Harris Corners Detected: {harris_count}")
    print(f"Shi-Tomasi Corners Detected: {shi_count}")
    print(f"Difference (Harris - Shi): {harris_count - shi_count}")


Image: shape1.jpg
Harris Corners Detected: 3736
Shi-Tomasi Corners Detected: 100
Difference (Harris - Shi): 3636

Image: building1.jpg
Harris Corners Detected: 46733
Shi-Tomasi Corners Detected: 100
Difference (Harris - Shi): 46633

Image: shape5.jpg
Harris Corners Detected: 23740
Shi-Tomasi Corners Detected: 100
Difference (Harris - Shi): 23640

Image: building3.jpg
Harris Corners Detected: 299970
Shi-Tomasi Corners Detected: 100
Difference (Harris - Shi): 299870

Image: building2.jpg
Harris Corners Detected: 584907
Shi-Tomasi Corners Detected: 100
Difference (Harris - Shi): 584807

Image: shape4.jpg
Harris Corners Detected: 750541
Shi-Tomasi Corners Detected: 100
Difference (Harris - Shi): 750441

Image: grid3.jpg
Harris Corners Detected: 1192
Shi-Tomasi Corners Detected: 100
Difference (Harris - Shi): 1092

Image: grid2.jpg
Harris Corners Detected: 54577
Shi-Tomasi Corners Detected: 100
Difference (Harris - Shi): 54477

Image: object1.jpg
Harris Corners Detected: 3693
Shi-Tomasi Co

## 5. Parameter Sensitivity Analysis

Corner detection performance depends on algorithm parameters. 

- **Harris (`k`)**: Increasing `k` reduces the number of detected corners by filtering weaker responses, keeping only the strongest.  
- **Shi–Tomasi (`qualityLevel`)**: Increasing `qualityLevel` raises the minimum quality for corners, reducing the number of detected points.  
- `max_corners` sets an upper limit on Shi–Tomasi detections to ensure fair comparison with Harris.

This analysis helps us choose **optimal parameters** for stable and meaningful corner detection.

In [9]:
# Use first image for parameter testing
sample_path = os.path.join(IMAGE_DIR, image_files[0])
image, blur = preprocess_image(sample_path)

print("\nHarris Parameter Sensitivity:")
for k in [0.04, 0.05, 0.06]:
    _, count = harris_corner_detection(image.copy(), blur, k=k)
    print(f"k = {k} → corners = {count}")


Harris Parameter Sensitivity:
k = 0.04 → corners = 3736
k = 0.05 → corners = 3088
k = 0.06 → corners = 2635


In [12]:
print("\nShi-Tomasi Parameter Sensitivity:")
for q in [0.01, 0.03, 0.05]:
    _, count = shi_tomasi_detection(image.copy(),blur,max_corners=1000, quality=q)
    print(f"qualityLevel = {q} → corners = {count}")


Shi-Tomasi Parameter Sensitivity:
qualityLevel = 0.01 → corners = 1000
qualityLevel = 0.03 → corners = 417
qualityLevel = 0.05 → corners = 294


## 6. Observations

From our experiments and parameter analysis, we observe the following:

- **Harris Corner Detection** generally detects a higher number of corner points. This includes both strong and weak corners, making it more sensitive to image details and noise.
- **Shi–Tomasi Corner Detection** selects fewer corners, but they tend to be stronger and more stable. This demonstrates its reliability for tracking and robust feature extraction.
- **Effect of thresholds and parameters**: Increasing the Harris `k` value or the Shi–Tomasi `qualityLevel` reduces the number of detected corners, effectively filtering out weaker points.
- **Preprocessing impact**: Applying grayscale conversion and Gaussian smoothing significantly improves corner detection stability and reduces false positives.

## 7. Conclusion

Both Harris and Shi–Tomasi methods are effective for detecting corner features in real-world images.  

- **Harris** is useful when detecting a large number of corners is desired, including weaker features for detailed analysis.  
- **Shi–Tomasi** provides stronger, more reliable corners, making it the preferred choice for applications that require robustness, such as feature tracking or shape analysis.  

Careful **parameter tuning** and preprocessing are essential for maximizing detection quality and minimizing noise.